# RMSD of CESA3 — merged trajectory (T1→T2→T3→T4) vs common starting structure

Computes backbone RMSD of the full concatenated, cross-fitted trajectory
(`CESA3_all_backbone.xtc`) against the single shared reference structure
(`R1/backbone.pdb` — frame 0 of T1), produced by `process_trajectories_CESA3.sh`.

This answers: **"How far has the protein moved from one common starting
point, across all 4 independent runs?"** — as a single continuous curve
spanning T1, T2, T3, T4 back to back (~1600 ns total, ~400 ns each).

**Inputs (already produced by `process_trajectories_CESA3.sh`):**
- Reference: `R1/backbone.pdb`
- Merged trajectory: `CESA3_all_backbone.xtc` (T1+T2+T3+T4, fit to R1/backbone.pdb)

**Outputs (written next to this notebook):**
- `rmsd_merged_complex.csv` — per-frame RMSD, with a `segment` column (T1/T2/T3/T4)
- `rmsd_merged_complex_summary.csv` — mean / std / min / max overall + per segment
- `rmsd_merged_complex.png` — RMSD vs time plot, with segment boundaries marked

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.analysis import rms

print('MDAnalysis', mda.__version__)

## 1. Configuration

In [ ]:
# BASE_DIR = the folder containing R1/ and CESA3_all_backbone.xtc
BASE_DIR = BASE_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')

REF_PDB    = BASE_DIR / 'R1' / 'backbone.pdb'          # reference structure (T1, frame 0)
MERGED_XTC = BASE_DIR / 'CESA3_all_backbone.xtc'        # T1+T2+T3+T4, fit to REF_PDB
ATOM_SEL   = 'backbone'                                 # MDAnalysis selection used for RMSD

# Trajectory lengths (ns) used to label/segment the merged file.
# These must match what process_trajectories_CESA3.sh actually produced for you:
# 450 ns runs minus the first 50 ns equilibration cut = ~400 ns each.
SEGMENT_NAMES    = ['T1', 'T2', 'T3', 'T4']
SEGMENT_DURATION_NS = 400.0   # adjust if your actual per-T length differs

# Output files (written next to this notebook)
OUT_DIR        = BASE_DIR
OUT_CSV_FRAMES = OUT_DIR / 'rmsd_merged_complex.csv'
OUT_CSV_STATS  = OUT_DIR / 'rmsd_merged_complex_summary.csv'
OUT_PNG        = OUT_DIR / 'rmsd_merged_complex.png'

assert REF_PDB.exists(), f'Reference PDB not found: {REF_PDB}'
assert MERGED_XTC.exists(), f'Missing trajectory: {MERGED_XTC}'
print(f'Reference : {REF_PDB.relative_to(BASE_DIR)}')
print(f'Trajectory: {MERGED_XTC.relative_to(BASE_DIR)}  '
      f'({MERGED_XTC.stat().st_size / 1e6:.1f} MB)')

## 2. Compute RMSD for the whole merged trajectory

`MDAnalysis.analysis.rms.RMSD` performs an optimal rigid-body superposition
of the selection on the reference at every frame and reports RMSD in Angstrom.
Since `CESA3_all_backbone.xtc` was already fit (rot+trans) to `R1/backbone.pdb`
in the processing pipeline, this is computing RMSD on top of an
already-aligned trajectory — consistent with how it was built.

In [ ]:
u   = mda.Universe(str(REF_PDB), str(MERGED_XTC))
ref = mda.Universe(str(REF_PDB))

rmsd_calc = rms.RMSD(u, ref, select=ATOM_SEL).run()
arr = rmsd_calc.results.rmsd          # columns: frame, time(ps), rmsd(A)

frame_idx = arr[:, 0].astype(int)
t_ns      = arr[:, 1] / 1000.0        # ps -> ns
r_a       = arr[:, 2]                 # Angstrom

print(f'Loaded {len(arr)} frames, '
      f'total span {t_ns[-1]-t_ns[0]:.1f} ns, '
      f'mean RMSD={r_a.mean():.3f} A  std={r_a.std():.3f} A')

## 3. Label each frame with its source trajectory (T1/T2/T3/T4)

`gmx trjcat -settime` lays the four segments back-to-back on a continuous
time axis, so segment boundaries are inferred here purely from elapsed time
(0–400 ns = T1, 400–800 ns = T2, etc.) — adjust `SEGMENT_DURATION_NS` above
if your actual per-trajectory length differs.

In [ ]:
t_rel = t_ns - t_ns[0]   # time relative to the start of T1
segment = np.array([
    SEGMENT_NAMES[min(int(t // SEGMENT_DURATION_NS), len(SEGMENT_NAMES) - 1)]
    for t in t_rel
])

df_frames = pd.DataFrame({
    'frame'    : frame_idx,
    'time_ns'  : t_ns,
    'rmsd_A'   : r_a,
    'segment'  : segment,
})
df_frames.to_csv(OUT_CSV_FRAMES, index=False)
print(f'Wrote {OUT_CSV_FRAMES.name}  ({len(df_frames):,} rows)')
df_frames.head()

## 4. Summary table — overall and per segment

In [ ]:
summary_rows = []
for seg in SEGMENT_NAMES:
    sub = df_frames.loc[df_frames['segment'] == seg, 'rmsd_A']
    if len(sub) == 0:
        continue
    summary_rows.append({
        'segment' : seg,
        'n_frames': len(sub),
        'mean_A'  : float(sub.mean()),
        'std_A'   : float(sub.std()),
        'min_A'   : float(sub.min()),
        'max_A'   : float(sub.max()),
    })

summary_rows.append({
    'segment' : 'OVERALL (full merged trajectory)',
    'n_frames': len(df_frames),
    'mean_A'  : float(df_frames['rmsd_A'].mean()),
    'std_A'   : float(df_frames['rmsd_A'].std()),
    'min_A'   : float(df_frames['rmsd_A'].min()),
    'max_A'   : float(df_frames['rmsd_A'].max()),
})

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_CSV_STATS, index=False)
print(f'Wrote {OUT_CSV_STATS.name}')
df_summary.round({'mean_A': 3, 'std_A': 3, 'min_A': 3, 'max_A': 3})

## 5. Plot

A single continuous RMSD curve across the full merged trajectory, with
dashed vertical lines marking where one independent run ends and the next
begins (T1→T2, T2→T3, T3→T4). Jumps at these lines reflect genuine
differences between independent simulations, not processing artifacts —
see earlier discussion on cross-trajectory fitting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(df_frames['time_ns'], df_frames['rmsd_A'],
        color='#1f77b4', lw=0.9, alpha=0.85)

# Mark segment boundaries
boundary_times = [i * SEGMENT_DURATION_NS for i in range(1, len(SEGMENT_NAMES))]
for bt in boundary_times:
    ax.axvline(bt, color='grey', ls='--', lw=0.8, alpha=0.7)

# Label each segment at the top of the plot
y_top = df_frames['rmsd_A'].max() * 1.05
for i, seg in enumerate(SEGMENT_NAMES):
    x_center = (i + 0.5) * SEGMENT_DURATION_NS
    ax.text(x_center, y_top, seg, ha='center', va='bottom',
            fontsize=10, color='grey')

overall_mean = df_frames['rmsd_A'].mean()
ax.axhline(overall_mean, color='black', lw=1.2, ls=':', alpha=0.6,
           label=f'overall mean = {overall_mean:.2f} A')

ax.set_xlabel('Time (ns)')
ax.set_ylabel('Backbone RMSD (A)')
ax.set_title('CESA3 — backbone RMSD vs common starting structure (R1/T1, t=0)\n'
             'Merged trajectory: T1 -> T2 -> T3 -> T4')
ax.legend(loc='best', frameon=False, fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_PNG, dpi=200)
print(f'Wrote {OUT_PNG.name}')
plt.show()

## 6. Headline number

In [ ]:
print(f'Backbone RMSD of the merged CESA3 trajectory '
      f'(vs common starting structure, {t_ns[0]:.1f}-{t_ns[-1]:.1f} ns):')
print(f'    {df_frames["rmsd_A"].mean():.3f} +/- {df_frames["rmsd_A"].std():.3f} A  (overall)')
print()
print('Per-segment (per independent trajectory):')
for _, row in df_summary.iterrows():
    if row['segment'] == 'OVERALL (full merged trajectory)':
        continue
    print(f"    {row['segment']}: {row['mean_A']:.3f} +/- {row['std_A']:.3f} A")